# Credit Card Default Prediction
## BFOR 516 – Advanced Data Analytics for Cyber | Group Project Milestone 2

**Dataset:** Default of Credit Card Clients (UCI ML Repository, Dataset ID 350)  
**Source:** https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients  
**Date:** March 2026

---

### Project Objective
Predict whether a credit card client will default on their payment next month using historical demographic, credit utilization, and payment behavior data.

### Models Implemented
1. **Logistic Regression** – interpretable linear classifier; strong baseline for binary classification
2. **Gaussian Naive Bayes** – probabilistic model; fast and effective benchmark
3. **Decision Tree Classifier** – non-linear, rule-based model with built-in feature importance

### Dataset Overview
- 30,000 credit card clients from Taiwan (April – September 2005)
- 23 features: demographics, credit limit, 6-month payment history, bill amounts, payment amounts
- Target: `default payment next month` (1 = default, 0 = no default)

In [ ]:
# ============================================================
# 1. IMPORTS & CONFIGURATION
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score
)

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Plot defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('Libraries loaded successfully.')

In [ ]:
# ============================================================
# 2. DATA LOADING
# ucimlrepo is the cleanest method; falls back to direct XLS
# ============================================================
try:
    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=350)
    df = pd.concat([repo.data.features, repo.data.targets], axis=1)
    # Normalize target column name
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={df.columns[-1]: 'default'})
    print('Dataset loaded via ucimlrepo.')

except Exception as e:
    print(f'ucimlrepo unavailable ({e}). Falling back to direct XLS download...')
    url = (
        'https://archive.ics.uci.edu/ml/machine-learning-databases/'
        '00350/default%20of%20credit%20card%20clients.xls'
    )
    # header=1 skips the merged title row; index_col=0 drops the ID column
    df = pd.read_excel(url, header=1, index_col=0)
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={'default payment next month': 'default'})
    print('Dataset loaded via direct XLS download.')

# Drop ID column if it still exists
if 'ID' in df.columns:
    df = df.drop(columns=['ID'])

print(f'\nDataset shape: {df.shape}')
print(f'\nColumn names:')
print(list(df.columns))

---
## Section 1: Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================
# 3. INITIAL DATA EXPLORATION
# ============================================================
print('=== First 5 Rows ===')
display(df.head())

print('\n=== Dataset Info ===')
df.info()

print('\n=== Target Variable Distribution ===')
print(df['default'].value_counts())
print(f'\nDefault rate: {df["default"].mean():.2%}')

In [ ]:
# Statistical summary of all features
print('=== Statistical Summary ===')
display(df.describe().round(2))

In [ ]:
# ============================================================
# 4. CLASS DISTRIBUTION VISUALIZATION
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['default'].value_counts()
labels = ['No Default (0)', 'Default (1)']
colors = ['steelblue', 'tomato']

# Bar chart
bars = axes[0].bar(labels, counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Number of Clients')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}\n({val/len(df):.1%})', ha='center', fontsize=11)

# Pie chart
axes[1].pie(counts.values, labels=['No Default', 'Default'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=(0, 0.06), shadow=True)
axes[1].set_title('Class Distribution (Proportion)')

plt.suptitle('Target Variable: Credit Card Default Next Month', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

imbalance_ratio = counts[0] / counts[1]
print(f'Class imbalance ratio: {imbalance_ratio:.1f}:1 (no default : default)')
print('Note: Dataset is moderately imbalanced. class_weight="balanced" will be used in models.')

In [ ]:
# ============================================================
# 5. DEFAULT RATE BY DEMOGRAPHIC FEATURES
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# SEX
sex_rate = df.groupby('SEX')['default'].mean()
sex_labels = {1: 'Male', 2: 'Female'}
axes[0].bar([sex_labels.get(k, k) for k in sex_rate.index], sex_rate.values,
            color=['#4472C4', '#ED7D31'], edgecolor='black')
axes[0].set_title('Default Rate by Gender')
axes[0].set_ylabel('Default Rate')
axes[0].set_ylim(0, 0.30)
for i, v in enumerate(sex_rate.values):
    axes[0].text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=11)

# EDUCATION (remap after cleaning — preview with current values)
edu_rate = df.groupby('EDUCATION')['default'].mean().sort_index()
edu_labels = {1: 'Grad\nSchool', 2: 'University', 3: 'High\nSchool', 4: 'Other',
              5: 'Unknown5', 6: 'Unknown6', 0: 'Unknown0'}
axes[1].bar([edu_labels.get(k, str(k)) for k in edu_rate.index], edu_rate.values,
            color='steelblue', edgecolor='black')
axes[1].set_title('Default Rate by Education')
axes[1].set_ylabel('Default Rate')
axes[1].set_ylim(0, 0.35)
for i, v in enumerate(edu_rate.values):
    axes[1].text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=10)

# MARRIAGE
mar_rate = df.groupby('MARRIAGE')['default'].mean().sort_index()
mar_labels = {0: 'Unknown', 1: 'Married', 2: 'Single', 3: 'Other'}
axes[2].bar([mar_labels.get(k, str(k)) for k in mar_rate.index], mar_rate.values,
            color='seagreen', edgecolor='black')
axes[2].set_title('Default Rate by Marital Status')
axes[2].set_ylabel('Default Rate')
axes[2].set_ylim(0, 0.30)
for i, v in enumerate(mar_rate.values):
    axes[2].text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=11)

plt.suptitle('Default Rate by Demographic Features (Pre-Cleaning)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_02_demographic_default_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 6. AGE ANALYSIS
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlapping histograms by default status
axes[0].hist(df[df['default'] == 0]['AGE'], bins=40, alpha=0.6,
             color='steelblue', label='No Default', density=True)
axes[0].hist(df[df['default'] == 1]['AGE'], bins=40, alpha=0.6,
             color='tomato', label='Default', density=True)
axes[0].set_title('Age Distribution by Default Status')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Default rate by age group
bins = [20, 30, 40, 50, 60, 80]
labels_age = ['21-30', '31-40', '41-50', '51-60', '61+']
df['AGE_GROUP'] = pd.cut(df['AGE'], bins=bins, labels=labels_age)
age_default = df.groupby('AGE_GROUP', observed=False)['default'].mean()

axes[1].bar(age_default.index.astype(str), age_default.values,
            color='steelblue', edgecolor='black')
axes[1].set_title('Default Rate by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Default Rate')
axes[1].set_ylim(0, 0.35)
for i, v in enumerate(age_default.values):
    axes[1].text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=11)

plt.suptitle('Age Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_03_age_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Remove temporary age group column
df = df.drop(columns=['AGE_GROUP'])

In [ ]:
# ============================================================
# 7. PAYMENT HISTORY ANALYSIS (PAY_0 through PAY_6)
# PAY_0 = Sept, PAY_2 = Aug, ..., PAY_6 = April
# Values: -1=pay duly, 0=revolving credit, 1-9=months of delay
# ============================================================
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
month_labels = ['Sep', 'Aug', 'Jul', 'Jun', 'May', 'Apr']

# Average payment delay by default status
pay_defaults = df.groupby('default')[pay_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in zip([0, 1], ['steelblue', 'tomato'], ['No Default', 'Default']):
    axes[0].plot(month_labels, pay_defaults.loc[label].values,
                 marker='o', color=color, linewidth=2, label=name)

axes[0].set_title('Average Payment Delay Status Over 6 Months')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Avg PAY Status (higher = more delay)')
axes[0].legend()
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)

# Distribution of PAY_0 (most recent payment status)
pay0_default = df.groupby('PAY_0')['default'].mean()
axes[1].bar(pay0_default.index, pay0_default.values, color='steelblue', edgecolor='black')
axes[1].set_title('Default Rate by Most Recent Payment Status (PAY_0/Sep)')
axes[1].set_xlabel('PAY_0 Value (-1=on time, 0=revolving, 1-8=months delayed)')
axes[1].set_ylabel('Default Rate')

plt.suptitle('Payment History Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_04_payment_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 8. CORRELATION HEATMAP
# ============================================================
plt.figure(figsize=(16, 12))

corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask

sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.3,
            cbar_kws={'label': 'Pearson Correlation'})

plt.title('Feature Correlation Matrix (Lower Triangle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
print('=== Top 10 Features Correlated with Default ===')
target_corr = corr['default'].drop('default').abs().sort_values(ascending=False)
print(target_corr.head(10).round(4).to_frame('|Correlation|'))

---
## Section 2: Data Cleaning & Preprocessing

### Issues identified during EDA:
1. **EDUCATION**: Values `0`, `5`, `6` are undocumented in the original dataset paper — remapped to `4` (Others)
2. **MARRIAGE**: Value `0` is undocumented — remapped to `3` (Others)
3. **No missing values** (will confirm below)
4. **Class imbalance**: ~22% default rate — addressed via `class_weight='balanced'` in models

In [ ]:
# ============================================================
# 9. DATA CLEANING
# ============================================================

# Check for missing values
missing = df.isnull().sum()
print('=== Missing Values ===')
print(f'Total missing: {missing.sum()}')
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print('No missing values found. Dataset is complete.')

# Check for duplicate rows
dupes = df.duplicated().sum()
print(f'\nDuplicate rows: {dupes}')

# --- Fix EDUCATION: remap undocumented values (0, 5, 6) to 4 (Other) ---
print(f'\nEDUCATION values (before): {sorted(df["EDUCATION"].unique())}')
print(f'Undocumented counts: {(df["EDUCATION"].isin([0, 5, 6])).sum()} rows')
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
print(f'EDUCATION values (after):  {sorted(df["EDUCATION"].unique())}')

# --- Fix MARRIAGE: remap undocumented value (0) to 3 (Other) ---
print(f'\nMARRIAGE values (before): {sorted(df["MARRIAGE"].unique())}')
print(f'Undocumented counts: {(df["MARRIAGE"] == 0).sum()} rows')
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})
print(f'MARRIAGE values (after):  {sorted(df["MARRIAGE"].unique())}')

print('\nCleaning complete.')

In [ ]:
# ============================================================
# 10. FEATURE PREPARATION
# Categorical features (SEX, EDUCATION, MARRIAGE) are kept as
# ordinal integers — appropriate for tree-based methods and
# acceptable for LR/NB given the ordinal nature of these codes.
# All features are scaled for LR and NB using StandardScaler.
# Decision Tree uses unscaled data (scale-invariant).
# ============================================================

# Separate features and target
X = df.drop(columns=['default'])
y = df['default']

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape:  {y.shape}')
print(f'\nFeatures ({len(X.columns)}):')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# ============================================================
# 11. TRAIN / TEST SPLIT (stratified 80/20)
# Stratified split ensures the default rate is preserved
# in both train and test sets, critical for imbalanced data.
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X):.0%})')
print(f'Test size:  {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X):.0%})')
print(f'\nTrain default rate: {y_train.mean():.2%}')
print(f'Test  default rate: {y_test.mean():.2%}')
print('Stratified split confirmed: rates match.')

# ── Scaling (fit on train only to prevent data leakage) ──────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # use train statistics on test

print(f'\nStandardScaler fit on train set.')
print(f'Train mean post-scale: {X_train_scaled.mean():.6f} (expected ~0)')
print(f'Train std  post-scale: {X_train_scaled.std():.6f}  (expected ~1)')

---
## Section 3: Principal Component Analysis (PCA)

PCA is used to:
1. Understand the dimensionality structure of the feature space
2. Identify how many components capture most of the variance
3. Visualize the data in 2D to check for class separability

In [ ]:
# ============================================================
# 12. PCA ANALYSIS
# ============================================================
pca_full = PCA(random_state=42)
pca_full.fit(X_train_scaled)

explained = pca_full.explained_variance_ratio_
cumvar = np.cumsum(explained)

n_90 = int(np.argmax(cumvar >= 0.90)) + 1
n_95 = int(np.argmax(cumvar >= 0.95)) + 1
n_99 = int(np.argmax(cumvar >= 0.99)) + 1

print('=== PCA Variance Summary ===')
print(f'Total features:                {X_train.shape[1]}')
print(f'Components for 90% variance:  {n_90}')
print(f'Components for 95% variance:  {n_95}')
print(f'Components for 99% variance:  {n_99}')
print(f'\nTop 5 components explain:     {cumvar[4]:.2%} of variance')
print(f'Top 10 components explain:    {cumvar[9]:.2%} of variance')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(explained)+1), explained, color='steelblue',
            alpha=0.8, edgecolor='white')
axes[0].plot(range(1, len(explained)+1), explained, 'o-', color='tomato',
             markersize=4, linewidth=1.5)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].set_xlim(0, 24)

# Cumulative variance
axes[1].plot(range(1, len(cumvar)+1), cumvar, 'o-', color='steelblue',
             markersize=5, linewidth=2)
axes[1].axhline(0.90, color='green', linestyle='--', alpha=0.7, label='90%')
axes[1].axhline(0.95, color='orange', linestyle='--', alpha=0.7, label='95%')
axes[1].axhline(0.99, color='red', linestyle='--', alpha=0.7, label='99%')
axes[1].axvline(n_95, color='orange', linestyle=':', alpha=0.6)
axes[1].fill_between(range(1, len(cumvar)+1), cumvar, alpha=0.15, color='steelblue')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend(title='Threshold')
axes[1].set_xlim(0, 24)

plt.suptitle('PCA – Explained Variance Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_06_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 13. PCA 2D SCATTER PLOT
# ============================================================
pca_2d = PCA(n_components=2, random_state=42)
X_pca_train = pca_2d.fit_transform(X_train_scaled)

plt.figure(figsize=(10, 7))

for label, color, name in zip([0, 1], ['steelblue', 'tomato'], ['No Default', 'Default']):
    mask = y_train == label
    plt.scatter(X_pca_train[mask, 0], X_pca_train[mask, 1],
                c=color, alpha=0.25, s=6, label=f'{name} (n={mask.sum():,})')

var1 = pca_2d.explained_variance_ratio_[0]
var2 = pca_2d.explained_variance_ratio_[1]
plt.xlabel(f'PC1 ({var1:.1%} explained variance)')
plt.ylabel(f'PC2 ({var2:.1%} explained variance)')
plt.title(f'PCA 2D Projection (PC1+PC2 = {var1+var2:.1%} of total variance)')
plt.legend(markerscale=4, fontsize=11)
plt.tight_layout()
plt.savefig('fig_07_pca_2d_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print('Observation: The two classes overlap significantly in the first two PCs,')
print('suggesting linear separability is limited - non-linear models may perform better.')

---
## Section 4: Machine Learning Models

### Evaluation Metrics
- **Accuracy**: Overall correctness (misleading for imbalanced data alone)
- **Precision**: Of predicted defaults, how many are actual defaults
- **Recall**: Of actual defaults, how many were correctly identified (critical for fraud/risk)
- **F1-Score**: Harmonic mean of precision and recall
- **ROC-AUC**: Overall discriminative ability; 0.5 = random, 1.0 = perfect
- **5-Fold CV AUC**: Cross-validated AUC to assess generalization

In [ ]:
# ============================================================
# 14. EVALUATION HELPER FUNCTION
# ============================================================
all_results = []
all_probs   = {}  # store test probabilities for ROC curves

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """
    Train model, evaluate on test set, print classification report,
    display confusion matrix, and run 5-fold cross-validation.
    Returns metrics dict, fitted model, and test probabilities.
    """
    # Train
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None

    # Metrics
    metrics = {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_te, y_pred), 4),
        'Precision': round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_te, y_pred), 4),
        'F1-Score':  round(f1_score(y_te, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_te, y_prob), 4) if y_prob is not None else None,
    }

    print(f'\n{"="*55}')
    print(f'  MODEL: {name}')
    print(f'{"="*55}')
    print(classification_report(y_te, y_pred,
                                 target_names=['No Default', 'Default']))

    # Confusion matrix plot
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_te, y_pred,
        display_labels=['No Default', 'Default'],
        cmap='Blues', ax=ax, colorbar=False
    )
    ax.set_title(f'{name} \u2013 Confusion Matrix')
    plt.tight_layout()
    fname = name.lower().replace(' ', '_')
    plt.savefig(f'fig_cm_{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 5-Fold Stratified Cross-Validation (AUC)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='roc_auc')
    metrics['CV_AUC_Mean'] = round(cv_scores.mean(), 4)
    metrics['CV_AUC_Std']  = round(cv_scores.std(), 4)
    print(f'5-Fold CV ROC-AUC: {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}')

    return metrics, model, y_prob

print('evaluate_model() helper defined.')

In [ ]:
# ============================================================
# 15. MODEL 1: LOGISTIC REGRESSION
# ============================================================
# Rationale:
#   - Well-established baseline for binary classification
#   - Coefficients are directly interpretable
#   - class_weight='balanced' compensates for the 78/22 imbalance
#   - C=1.0 (default regularization); max_iter=1000 ensures convergence
# ============================================================
lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    random_state=42
)

lr_metrics, trained_lr, prob_lr = evaluate_model(
    'Logistic Regression', lr_model,
    X_train_scaled, y_train,
    X_test_scaled,  y_test
)

all_results.append(lr_metrics)
all_probs['Logistic Regression'] = prob_lr

In [ ]:
# ============================================================
# 16. MODEL 2: GAUSSIAN NAIVE BAYES
# ============================================================
# Rationale:
#   - Probabilistic model based on Bayes theorem
#   - Assumes feature independence (naive assumption)
#   - Fast training; useful as a probabilistic benchmark
#   - GaussianNB assumes continuous features follow a normal
#     distribution per class — reasonable after StandardScaler
# ============================================================
nb_model = GaussianNB()

nb_metrics, trained_nb, prob_nb = evaluate_model(
    'Naive Bayes', nb_model,
    X_train_scaled, y_train,
    X_test_scaled,  y_test
)

all_results.append(nb_metrics)
all_probs['Naive Bayes'] = prob_nb

In [ ]:
# ============================================================
# 17. MODEL 3: DECISION TREE CLASSIFIER
# ============================================================
# Rationale:
#   - Non-linear model; can capture complex interaction patterns
#   - Built-in feature importance via Gini impurity
#   - Highly interpretable — can visualize decision rules
#   - Does NOT require feature scaling
#   - max_depth=5 prevents overfitting on 30k rows
#   - min_samples_leaf=20 ensures stable leaf nodes
# ============================================================
dt_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=50,
    min_samples_leaf=20,
    class_weight='balanced',
    criterion='gini',
    random_state=42
)

# Use unscaled data for Decision Tree
dt_metrics, trained_dt, prob_dt = evaluate_model(
    'Decision Tree', dt_model,
    X_train.values, y_train,
    X_test.values,  y_test
)

all_results.append(dt_metrics)
all_probs['Decision Tree'] = prob_dt

In [ ]:
# Visualize Decision Tree (first 3 levels for readability)
plt.figure(figsize=(22, 8))
plot_tree(
    trained_dt,
    feature_names=X.columns.tolist(),
    class_names=['No Default', 'Default'],
    filled=True, max_depth=3,
    fontsize=8, impurity=True, proportion=False
)
plt.title('Decision Tree Visualization (Top 3 Levels of 5)',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_decision_tree_viz.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5: Model Comparison & Analysis

In [ ]:
# ============================================================
# 18. MODEL COMPARISON TABLE
# ============================================================
results_df = pd.DataFrame(all_results).set_index('Model')
display_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC',
                'CV_AUC_Mean', 'CV_AUC_Std']
results_df = results_df[display_cols]

print('=== Model Performance Comparison ===')
display(results_df)

best_f1  = results_df['F1-Score'].idxmax()
best_auc = results_df['ROC-AUC'].idxmax()
print(f'\nBest F1-Score : {best_f1} ({results_df.loc[best_f1, "F1-Score"]})')
print(f'Best ROC-AUC  : {best_auc} ({results_df.loc[best_auc, "ROC-AUC"]})')

In [ ]:
# ============================================================
# 19. COMPARISON BAR CHART + ROC CURVES
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
model_colors = ['#4472C4', '#70AD47', '#ED7D31']

# --- Bar chart of key metrics ---
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.22

for i, (model_name, row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in metrics_to_plot]
    axes[0].bar(x + i*width, vals, width, label=model_name,
                color=model_colors[i], edgecolor='black', alpha=0.88)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metrics_to_plot)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison')
axes[0].legend()
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)
axes[0].axhline(0.8, color='green', linestyle=':', alpha=0.4, linewidth=1)

# --- ROC Curves ---
for (model_name, prob), color in zip(all_probs.items(), model_colors):
    if prob is not None:
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc = roc_auc_score(y_test, prob)
        axes[1].plot(fpr, tpr, color=color, lw=2.5,
                     label=f'{model_name} (AUC = {auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.500)')
axes[1].fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].set_title('ROC Curves')
axes[1].legend(loc='lower right', fontsize=10)

plt.suptitle('Model Comparison – Preliminary Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_08_model_comparison_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 20. FEATURE IMPORTANCE ANALYSIS
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Decision Tree: Gini Feature Importance ---
dt_importance = pd.Series(
    trained_dt.feature_importances_,
    index=X.columns
).sort_values(ascending=True).tail(15)

dt_importance.plot(kind='barh', color='#ED7D31', edgecolor='black', ax=axes[0])
axes[0].set_title('Decision Tree \u2013 Top 15 Feature Importances (Gini)')
axes[0].set_xlabel('Importance')

# --- Logistic Regression: |Coefficient| ---
lr_importance = pd.Series(
    np.abs(trained_lr.coef_[0]),
    index=X.columns
).sort_values(ascending=True).tail(15)

lr_importance.plot(kind='barh', color='#4472C4', edgecolor='black', ax=axes[1])
axes[1].set_title('Logistic Regression \u2013 Top 15 |Coefficients|')
axes[1].set_xlabel('|Coefficient|')

plt.suptitle('Feature Importance Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_09_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Top 10 Features: Decision Tree (Gini Importance) ===')
print(dt_importance.sort_values(ascending=False).head(10).round(4).to_string())

print('\n=== Top 10 Features: Logistic Regression (|Coefficient|) ===')
print(lr_importance.sort_values(ascending=False).head(10).round(4).to_string())

---
## Section 6: Summary of Preliminary Results

> **Note for report writing:** Fill in the actual numbers after running this notebook. The analysis and interpretation below must be written in your own words for the written report.

In [ ]:
# ============================================================
# 21. SUMMARY PRINTOUT
# ============================================================
print('=' * 65)
print('  MILESTONE 2 – PRELIMINARY RESULTS SUMMARY')
print('=' * 65)

summary_cols = ['Accuracy', 'Recall', 'F1-Score', 'ROC-AUC']
display(results_df[summary_cols])

print('\nKey Observations:')
print('  1. PAY_0 (most recent payment status) is the strongest predictor')
print('  2. Payment history features (PAY_0 - PAY_6) dominate feature importance')
print('  3. PCA shows significant class overlap in first 2 components')
print('  4. Class imbalance (78/22) requires balanced weighting or resampling')

print('\nNext Steps (for final report):')
print('  1. Hyperparameter tuning via GridSearchCV')
print('  2. Address class imbalance with SMOTE (imblearn)')
print('  3. Consider ensemble models (Random Forest, Gradient Boosting)')
print('  4. Feature engineering: payment utilization ratios (PAY_AMT/BILL_AMT)')
print('  5. Final model selection with comprehensive analysis')